In [1]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "neo4j_password")

def query_neo4j(query, parameters=None):
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            result = session.run(query, parameters or {})
            return [dict(record) for record in result]

### Total nodes / relationships

In [4]:
query = """
CALL db.relationshipTypes() YIELD relationshipType
WHERE toLower(relationshipType) CONTAINS "treat"
RETURN relationshipType
"""

print(query_neo4j(query))

[]


In [5]:
query = """
MATCH ()-[r]->()
WHERE toLower(type(r)) CONTAINS "treat"
RETURN type(r) AS relationship_type,
       count(r) AS count
"""

print(query_neo4j(query))

[]


In [6]:
query = """
MATCH (n)
WHERE ANY(prop IN keys(n)
          WHERE toLower(toString(n[prop])) CONTAINS "treat")
RETURN labels(n) AS labels,
       n
LIMIT 25
"""

print(query_neo4j(query))

[{'labels': ['Journal'], 'n': <Node element_id='4:bd39a85b-13cb-497e-8194-630b76272c30:123' labels=frozenset({'Journal'}) properties={'journalId': 748, 'nlmTa': 'Neuropsychiatr Dis Treat', 'isoAbbrev': 'Neuropsychiatr Dis Treat', 'uri': 'http://example.org/ethnomed/journal/748', 'publisherIdentifier': 'NDT'}>}, {'labels': ['Journal'], 'n': <Node element_id='4:bd39a85b-13cb-497e-8194-630b76272c30:203' labels=frozenset({'Journal'}) properties={'journalId': 870, 'nlmTa': 'Depress Res Treat', 'isoAbbrev': 'Depress Res Treat', 'uri': 'http://example.org/ethnomed/journal/870', 'publisherIdentifier': 'DRT'}>}, {'labels': ['Journal'], 'n': <Node element_id='4:bd39a85b-13cb-497e-8194-630b76272c30:617' labels=frozenset({'Journal'}) properties={'journalId': 113, 'nlmTa': 'Schizophr Res Treatment', 'isoAbbrev': 'Schizophr Res Treatment', 'uri': 'http://example.org/ethnomed/journal/113', 'publisherIdentifier': 'SCHIZORT'}>}, {'labels': ['Article'], 'n': <Node element_id='4:bd39a85b-13cb-497e-8194-6

In [2]:
query = """
MATCH (n) RETURN count(n) AS total_nodes
"""
print(query_neo4j(query)[0])

query = """
MATCH ()-[r]->() RETURN count(r) AS total_relationships
"""
print(query_neo4j(query)[0])

{'total_nodes': 5012747}
{'total_relationships': 7709686}


In [7]:
query = """
MATCH ()-[r]->()
WHERE ANY(prop IN keys(r)
          WHERE toLower(toString(r[prop])) CONTAINS "treat")
RETURN type(r) AS rel_type,
       r
LIMIT 25
"""

print(query_neo4j(query))

[]


In [9]:
query = """
MATCH ()-[r:TREATS]->()
RETURN count(r) AS treat_count
"""

print(query_neo4j(query))

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `TREATS` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=13, offset=13>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 13, 'line': 2, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH ()-[r:TREATS]->()\nRETURN count(r) AS treat_count\n'


[{'treat_count': 0}]


### Labels + relationship types (top 20)

In [3]:
query = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS c
ORDER BY c DESC
LIMIT 20
"""
for row in query_neo4j(query):
    print(row)

query = """
MATCH ()-[r]->()
RETURN type(r) AS rel, count(*) AS c
ORDER BY c DESC
LIMIT 20
"""
for row in query_neo4j(query):
    print(row)

{'label': ['Resource', 'DbIpni', 'Name'], 'c': 1784251}
{'label': ['Resource'], 'c': 706870}
{'label': ['Species', 'Resource', 'DbWfo'], 'c': 381231}
{'label': ['Formulation'], 'c': 271000}
{'label': ['Resource', 'Compound', 'DbBRENDA'], 'c': 243928}
{'label': ['Resource', 'NameSynonym', 'DbChEBI', 'OtherName'], 'c': 240804}
{'label': ['Resource', 'DbChEBI', 'Compound'], 'c': 204833}
{'label': ['Resource', 'InChI'], 'c': 181064}
{'label': ['Resource', 'DbBRENDA', 'KmValue'], 'c': 172525}
{'label': ['Resource', 'DbChEBI', 'OtherName', 'NameIupacname'], 'c': 119231}
{'label': ['Resource', 'DbIpni', 'Location'], 'c': 76726}
{'label': ['Resource', 'DbBRENDA', 'IC50Value'], 'c': 76105}
{'label': ['Resource', 'DbBRENDA', 'NSPReaction'], 'c': 65402}
{'label': ['Resource', 'GeneralInformation', 'DbBRENDA'], 'c': 58965}
{'label': ['Resource', 'DbBRENDA', 'KcatKmValue'], 'c': 43326}
{'label': ['Resource', 'XrefBrendaligand'], 'c': 41634}
{'label': ['Resource', 'XrefSurechembl'], 'c': 38494}
{'la

## 1) “Readable fields” helpers (name/label/uri)

### Inspect properties for key labels

In [4]:
for lbl in ["Plant", "Compound", "Name", "Taxon", "Family"]:
    query = f"""
    MATCH (n:{lbl})
    RETURN keys(n) AS keys
    LIMIT 5
    """
    print(f"\n--- {lbl} keys ---")
    for row in query_neo4j(query):
        print(row)


--- Plant keys ---
{'keys': ['uri', 'taxon_name']}
{'keys': ['uri', 'taxon_name']}
{'keys': ['uri', 'taxon_name']}
{'keys': ['uri', 'taxon_name']}
{'keys': ['uri', 'taxon_name']}

--- Compound keys ---
{'keys': ['star', 'definition', 'source', 'uri', 'status_id', 'ascii_name']}
{'keys': ['star', 'definition', 'source', 'uri', 'status_id', 'ascii_name']}
{'keys': ['star', 'definition', 'source', 'uri', 'status_id', 'ascii_name']}
{'keys': ['star', 'definition', 'source', 'uri', 'status_id', 'ascii_name']}
{'keys': ['star', 'definition', 'source', 'uri', 'status_id', 'ascii_name']}

--- Name keys ---
{'keys': ['rank', 'name', 'uri']}
{'keys': ['rank', 'name', 'uri']}
{'keys': ['rank', 'name', 'uri']}
{'keys': ['rank', 'name', 'uri']}
{'keys': ['rank', 'name', 'uri']}

--- Taxon keys ---
{'keys': ['scientific_name', 'uri', 'rank']}
{'keys': ['scientific_name', 'uri', 'rank']}
{'keys': ['scientific_name', 'uri', 'rank']}
{'keys': ['scientific_name', 'uri', 'rank']}
{'keys': ['scientific_n

## 2) What does HAS_COMPOUND actually connect? 

### Discover HAS_COMPOUND endpoints (TOP 30)

In [5]:
query = """
MATCH (a)-[:HAS_COMPOUND]->(b)
RETURN labels(a) AS a_labels, labels(b) AS b_labels, count(*) AS c
ORDER BY c DESC
LIMIT 30
"""
for row in query_neo4j(query):
    print(row)

{'a_labels': ['Resource', 'DbCoconut', 'Organism'], 'b_labels': ['Resource', 'Compound', 'DbCoconut'], 'c': 1243891}
